# Alarm Episode Visualizations & Data Export

For each alarm episode, generate a folder containing:
1. **HTML plot**: All PV and OP tags (min-max normalized, original values on hover) with control actions subplot
2. **PV data CSV**: Minutewise PV/OP data for all available tags in the episode window
3. **Events CSV**: All CHANGE events occurring in the episode window

Each episode gets its own folder under `RESULTS/<tag>_cluster_visualizations/episode_XXXX/`.

Time window: `[episode_start - buffer, episode_end + buffer]`

**Configuration**: Edit cell 3 below to switch target tags.

In [5]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os
from pathlib import Path

In [6]:
# ═══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION — Change these to generate for a different tag
# ═══════════════════════════════════════════════════════════════════════════════

# Target tag
TARGET_TAG = '03LIC_1016'           # Base tag name (e.g., '03LIC_1071', '03LIC_1016')
TARGET_PV_COL = f'{TARGET_TAG}.PV'  # PV column name in time series data

# Alarm threshold — auto-detected from events AlarmLimit column
# Set manually to override, or leave as None for auto-detection
TARGET_ALARM_LIMIT = None           # None = auto-detect from events data

# Alarm condition to look up (used for auto-detecting alarm limit)
ALARM_CONDITION = 'PVLO'

# Data file paths
PV_OP_FILE = '../DATA/PV-OP_data/03LIC_1016_JAN_2026.parquet'           # PV/OP time series parquet
EVENTS_FILE = '/home/h604827/ControlActions/DATA/25_tags_events_preprocessed/51fda50f-24a3-4632-9803-7ec34744c34b.parquet'  # Events data

# Output directory — single folder per tag containing Excel + plots
RESULTS_DIR_NAME = f'{TARGET_TAG}_episodes'
CLUSTERS_FILE = f'../RESULTS/{RESULTS_DIR_NAME}/{TARGET_TAG}_{ALARM_CONDITION.lower()}_alarms_clustered_with_control_actions.xlsx'

# Trip filtering
TRIP_FILE = '../DATA/Final_List_Trip_Duration.csv'
FILTER_TRIPS = True

# Operating limits
OPERATING_LIMITS_FILE = '../DATA/operating_limits.xlsx'

# Plot settings
BUFFER_BEFORE_MINUTES = 240  # Minutes before cluster for plot window
BUFFER_AFTER_MINUTES = 60    # Minutes after cluster for plot window

print(f"Target: {TARGET_TAG} | Alarm limit: {TARGET_ALARM_LIMIT or 'auto-detect'}")
print(f"PV/OP data: {PV_OP_FILE}")
print(f"Events: {EVENTS_FILE}")
print(f"Clusters: {CLUSTERS_FILE}")
print(f"Trip filtering: {'ON' if FILTER_TRIPS else 'OFF'}")
print(f"Buffer: -{BUFFER_BEFORE_MINUTES} min / +{BUFFER_AFTER_MINUTES} min")
print(f"Output: RESULTS/{RESULTS_DIR_NAME}/")

Target: 03LIC_1016 | Alarm limit: auto-detect
PV/OP data: ../DATA/PV-OP_data/03LIC_1016_JAN_2026.parquet
Events: /home/h604827/ControlActions/DATA/25_tags_events_preprocessed/51fda50f-24a3-4632-9803-7ec34744c34b.parquet
Clusters: ../RESULTS/03LIC_1016_episodes/03LIC_1016_pvlo_alarms_clustered_with_control_actions.xlsx
Trip filtering: ON
Buffer: -240 min / +60 min
Output: RESULTS/03LIC_1016_episodes/


In [7]:
# Load data
DATA_DIR = Path('../DATA')
RESULTS_DIR = Path(f'../RESULTS/{RESULTS_DIR_NAME}')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ── Load PV/OP time series ──
op_pv_data_df = pd.read_parquet(PV_OP_FILE)
# Set TimeStamp as index if it's a column
if 'TimeStamp' in op_pv_data_df.columns:
    op_pv_data_df['TimeStamp'] = pd.to_datetime(op_pv_data_df['TimeStamp'])
    op_pv_data_df.set_index('TimeStamp', inplace=True)
op_pv_data_df.sort_index(inplace=True)

# ── Trip period filtering on PV/OP data ──
if FILTER_TRIPS:
    trips_df = pd.read_csv(TRIP_FILE)
    trips_df['Stop Date'] = pd.to_datetime(trips_df['Stop Date'])
    trips_df['Start Date'] = pd.to_datetime(trips_df['Start Date'])
    
    pre_trip = len(op_pv_data_df)
    trip_mask = pd.Series(False, index=op_pv_data_df.index)
    for _, trip in trips_df.iterrows():
        trip_mask |= (op_pv_data_df.index >= trip['Stop Date']) & (op_pv_data_df.index <= trip['Start Date'])
    op_pv_data_df = op_pv_data_df[~trip_mask]
    print(f'Trip filtering (PV/OP): {pre_trip:,} → {len(op_pv_data_df):,} rows (removed {pre_trip - len(op_pv_data_df):,})')

# ── Load alarm clusters and control actions ──
alarms_df = pd.read_excel(CLUSTERS_FILE, sheet_name=0)
actions_df = pd.read_excel(CLUSTERS_FILE, sheet_name=1)

# ── Load raw events (for custom window plots) ──
if EVENTS_FILE.endswith('.parquet'):
    raw_events_df = pd.read_parquet(EVENTS_FILE)
else:
    raw_events_df = pd.read_csv(EVENTS_FILE, low_memory=False)
raw_events_df['VT_Start'] = pd.to_datetime(raw_events_df['VT_Start'])
raw_events_df = raw_events_df.sort_values('VT_Start').reset_index(drop=True)

# ── Auto-detect alarm limit from events data ──
if TARGET_ALARM_LIMIT is None and 'AlarmLimit' in raw_events_df.columns:
    alarm_start_rows = raw_events_df[
        (raw_events_df['Source'] == TARGET_TAG) &
        (raw_events_df['ConditionName'] == ALARM_CONDITION) &
        (raw_events_df['Category'] == 1) &
        (raw_events_df['Action'].isna() | (raw_events_df['Action'] == ''))
    ]
    alarm_limits = alarm_start_rows['AlarmLimit'].dropna().unique()
    if len(alarm_limits) == 1:
        TARGET_ALARM_LIMIT = float(alarm_limits[0])
        print(f'Auto-detected alarm limit: {TARGET_ALARM_LIMIT} (from {len(alarm_start_rows)} {ALARM_CONDITION} alarm starts)')
    elif len(alarm_limits) > 1:
        TARGET_ALARM_LIMIT = float(alarm_limits[0])
        print(f'WARNING: Multiple alarm limits found {alarm_limits}, using first: {TARGET_ALARM_LIMIT}')
    else:
        print(f'Could not auto-detect alarm limit for {TARGET_TAG} {ALARM_CONDITION} (no matching rows with AlarmLimit)')

# Trip filter events
if FILTER_TRIPS:
    pre_trip_ev = len(raw_events_df)
    ev_trip_mask = pd.Series(False, index=raw_events_df.index)
    for _, trip in trips_df.iterrows():
        ev_trip_mask |= (raw_events_df['VT_Start'] >= trip['Stop Date']) & (raw_events_df['VT_Start'] <= trip['Start Date'])
    raw_events_df = raw_events_df[~ev_trip_mask].reset_index(drop=True)
    print(f'Trip filtering (events): {pre_trip_ev:,} → {len(raw_events_df):,} rows (removed {pre_trip_ev - len(raw_events_df):,})')

# Pre-filter to CHANGE events only
raw_change_events = raw_events_df[raw_events_df['ConditionName'] == 'CHANGE'].copy()

# ── Load operating limits ──
op_limits_raw = pd.read_excel(OPERATING_LIMITS_FILE)
tag_operating_limits = {}  # {tag_name: {'upper': val, 'lower': val}}
for _, row in op_limits_raw.iterrows():
    tag = row['TagName']
    upper = row['NEW_UPPER_LIMIT']
    lower = row['NEW_LOWER_LIMIT']
    if pd.isna(upper) or (isinstance(upper, str) and 'NOT' in upper.upper()):
        upper = row['OLD_UPPER_LIMIT']
    if pd.isna(lower) or (isinstance(lower, str) and 'NOT' in lower.upper()):
        lower = row['OLD_LOWER_LIMIT']
    try:
        upper = float(upper)
        lower = float(lower)
        tag_operating_limits[tag] = {'upper': upper, 'lower': lower}
    except (ValueError, TypeError):
        pass

# Target PV operating limits (from limits file)
TARGET_PV_LOWER = tag_operating_limits.get(TARGET_PV_COL, {}).get('lower', None)
TARGET_PV_UPPER = tag_operating_limits.get(TARGET_PV_COL, {}).get('upper', None)

print(f'\nPV/OP data: {op_pv_data_df.shape[0]:,} rows, {op_pv_data_df.index.min()} to {op_pv_data_df.index.max()}')
print(f'Alarm clusters: {alarms_df["cluster_id"].nunique()} clusters, {len(alarms_df)} individual alarms')
print(f'Control actions (Excel): {len(actions_df)} actions across {actions_df["cluster_id"].nunique()} clusters')
print(f'Raw CHANGE events: {len(raw_change_events):,}')
print(f'Operating limits loaded for {len(tag_operating_limits)} tags')
print(f'{TARGET_PV_COL} limits: lower={TARGET_PV_LOWER}, upper={TARGET_PV_UPPER}, alarm={TARGET_ALARM_LIMIT or "N/A"}')

Trip filtering (PV/OP): 1,737,586 → 1,718,039 rows (removed 19,547)


FileNotFoundError: [Errno 2] No such file or directory: '../RESULTS/03LIC_1016_episodes/03LIC_1016_pvlo_alarms_clustered_with_control_actions.xlsx'

In [ ]:
# Identify tag columns and group by base tag name
tag_cols = [c for c in op_pv_data_df.columns if c.endswith('.PV') or c.endswith('.OP')]
pv_cols = sorted([c for c in tag_cols if c.endswith('.PV')])
op_cols = sorted([c for c in tag_cols if c.endswith('.OP')])

# Build ordered list: for each base tag, PV first then OP
pv_bases = {c.replace('.PV', ''): c for c in pv_cols}
op_bases = {c.replace('.OP', ''): c for c in op_cols}
all_bases = sorted(set(list(pv_bases.keys()) + list(op_bases.keys())))

# Build ordered tag list grouped by base name
ordered_tags = []
for base in all_bases:
    if base in pv_bases:
        ordered_tags.append(pv_bases[base])
    if base in op_bases:
        ordered_tags.append(op_bases[base])

print(f'Total tags to plot: {len(ordered_tags)} ({len(pv_cols)} PV + {len(op_cols)} OP)')
print(f'Base tags: {len(all_bases)}')

Total tags to plot: 33 (18 PV + 15 OP)
Base tags: 22


In [ ]:
# Build cluster info: one row per cluster with start, end, and list of individual alarms
cluster_info = {}
for cid, grp in alarms_df.groupby('cluster_id'):
    cluster_info[cid] = {
        'cluster_start': grp['cluster_start_time'].iloc[0],
        'cluster_end': grp['cluster_end_time'].iloc[0],
        'cluster_type': grp['cluster_type'].iloc[0],
        'total_alarms': grp['cluster_total_alarms'].iloc[0],
        'alarms': list(grp[['alarm_start', 'alarm_end', 'episode_num']].itertuples(index=False, name=None)),
    }

# Build actions per cluster
cluster_actions = {}
for cid, grp in actions_df.groupby('cluster_id'):
    cluster_actions[cid] = grp.copy()

print(f'Total clusters: {len(cluster_info)}')
print(f'Clusters with actions: {len(cluster_actions)}')

Total clusters: 594
Clusters with actions: 296


In [ ]:
# Generate a consistent color palette for tags using plotly qualitative colors
import plotly.express as px

# Use a large qualitative palette
palette = (
    px.colors.qualitative.Dark24 +
    px.colors.qualitative.Light24 +
    px.colors.qualitative.Alphabet
)

# Assign colors by base tag (PV and OP of same base share same color)
base_colors = {}
for i, base in enumerate(all_bases):
    base_colors[base] = palette[i % len(palette)]

print(f'Color assignments for {len(base_colors)} base tags')

Color assignments for 22 base tags


In [ ]:
def create_window_plot(window_start, window_end, op_pv_df, ordered_tags, base_colors,
                       title='', actions=None, cluster_regions=None, alarm_regions=None,
                       target_limits=None, operating_limits=None,
                       target_tag=None, target_pv_col=None):
    """
    General-purpose plot for any time window.

    Parameters:
        window_start, window_end: pd.Timestamp - time window boundaries
        op_pv_df: DataFrame with TimeStamp index and PV/OP columns
        ordered_tags: list of column names to plot
        base_colors: dict mapping base tag name to color
        title: str - plot title
        actions: DataFrame of control actions (needs Source, Description, action_direction,
                 VT_Start, PrevValue, Value, action_timing columns). None if no actions.
        cluster_regions: list of (cluster_id, start, end) tuples for orange shading
        alarm_regions: list of (alarm_start, alarm_end, episode_num) tuples for red shading
        target_limits: dict with 'lower', 'upper', 'alarm' for target PV horizontal lines
        operating_limits: dict {tag_name: {'upper': val, 'lower': val}} for per-tag limit lines
        target_tag: str - base tag name to highlight (e.g., '03LIC_1016')
        target_pv_col: str - full PV column name (e.g., '03LIC_1016.PV')
    """
    # Use module-level defaults if not provided
    if target_tag is None:
        target_tag = TARGET_TAG
    if target_pv_col is None:
        target_pv_col = TARGET_PV_COL

    # Extract PV/OP data for window
    mask = (op_pv_df.index >= window_start) & (op_pv_df.index <= window_end)
    window_df = op_pv_df.loc[mask, [c for c in ordered_tags if c in op_pv_df.columns]].copy()

    if window_df.empty:
        print(f'  No PV/OP data in window {window_start} to {window_end}')
        return None

    has_actions = actions is not None and len(actions) > 0

    # Determine subplot heights
    if has_actions:
        n_action_tags = actions['Source'].nunique()
        action_height = max(0.15, min(0.35, n_action_tags * 0.04))
        row_heights = [1 - action_height, action_height]
        fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.05,
                            row_heights=row_heights,
                            subplot_titles=('PV / OP Trends (normalized)', 'Control Actions'))
    else:
        fig = make_subplots(rows=1, cols=1,
                            subplot_titles=('PV / OP Trends (normalized)',))

    # ── Top subplot: PV/OP traces (min-max normalized) ──
    x_range = [window_df.index[0], window_df.index[-1]]

    for col in ordered_tags:
        if col not in window_df.columns:
            continue
        series = window_df[col]
        if series.isna().all():
            continue

        col_min, col_max = series.min(), series.max()
        if col_max == col_min:
            normalized = pd.Series(0.5, index=series.index)
        else:
            normalized = (series - col_min) / (col_max - col_min)

        base_tag = col.rsplit('.', 1)[0]
        suffix = col.rsplit('.', 1)[1]
        color = base_colors.get(base_tag, '#888888')
        is_op = suffix == 'OP'
        is_target = base_tag == target_tag

        fig.add_trace(
            go.Scatter(
                x=series.index, y=normalized, mode='lines', name=col,
                legendgroup=base_tag,
                legendgrouptitle_text=base_tag if suffix == 'PV' else None,
                line=dict(color=color, width=2.5 if is_target else 1.5,
                          dash='dot' if is_op else 'solid'),
                visible=True if is_target else 'legendonly',
                customdata=np.column_stack([series.values]),
                hovertemplate=(f'<b>{col}</b><br>Time: %{{x}}<br>'
                               'Value: %{customdata[0]:.4f}<br><extra></extra>'),
            ),
            row=1, col=1
        )

        # ── Operating limit lines for this PV tag (upper + lower as single legend item) ──
        if operating_limits and suffix == 'PV' and col in operating_limits and col_max != col_min:
            limits = operating_limits[col]
            upper_val = limits['upper']
            lower_val = limits['lower']
            norm_upper = (upper_val - col_min) / (col_max - col_min)
            norm_lower = (lower_val - col_min) / (col_max - col_min)
            # Upper limit line (shows legend entry for both)
            fig.add_trace(
                go.Scatter(
                    x=x_range, y=[norm_upper, norm_upper], mode='lines',
                    name=f'{col} limits ({lower_val:.2f}–{upper_val:.2f})',
                    legendgroup=base_tag,
                    line=dict(color=color, width=1.2, dash='dash'),
                    visible=True if is_target else 'legendonly',
                    hovertemplate=f'<b>{col} upper limit</b><br>Value: {upper_val:.4f}<extra></extra>',
                ),
                row=1, col=1
            )
            # Lower limit line (no legend entry, tied to same group)
            fig.add_trace(
                go.Scatter(
                    x=x_range, y=[norm_lower, norm_lower], mode='lines',
                    name=f'{col} limits',
                    legendgroup=base_tag,
                    line=dict(color=color, width=1.2, dash='dash'),
                    visible=True if is_target else 'legendonly',
                    showlegend=False,
                    hovertemplate=f'<b>{col} lower limit</b><br>Value: {lower_val:.4f}<extra></extra>',
                ),
                row=1, col=1
            )

    # ── Horizontal limit lines for target PV (alarm threshold) ──
    if target_limits and target_pv_col in window_df.columns:
        pv_series = window_df[target_pv_col]
        pv_min, pv_max = pv_series.min(), pv_series.max()
        if pv_max != pv_min:
            # Only add alarm threshold line (operating limits already handled above)
            alarm_val = target_limits['alarm']
            norm_val = (alarm_val - pv_min) / (pv_max - pv_min)
            fig.add_trace(
                go.Scatter(
                    x=x_range, y=[norm_val, norm_val], mode='lines',
                    name=f'Alarm ({alarm_val})',
                    legendgroup=target_tag,
                    line=dict(color='red', width=1.5, dash='dash'),
                    visible=True,
                    hovertemplate=f'<b>{target_pv_col} alarm threshold</b><br>Value: {alarm_val}<extra></extra>',
                ),
                row=1, col=1
            )

    # ── Shading ──
    n_rows = 2 if has_actions else 1

    # Cluster-level shading (light orange)
    if cluster_regions:
        for cid_r, c_start_r, c_end_r in cluster_regions:
            for r in range(1, n_rows + 1):
                fig.add_vrect(x0=c_start_r, x1=c_end_r, fillcolor='rgba(255, 165, 0, 0.10)',
                              line=dict(width=0), layer='below', row=r, col=1)
                fig.add_vline(x=c_start_r, line=dict(color='orange', width=1.5, dash='dash'), row=r, col=1)
                fig.add_vline(x=c_end_r, line=dict(color='orange', width=1.5, dash='dash'), row=r, col=1)

    # Individual alarm shading (light red)
    if alarm_regions:
        for alarm_start, alarm_end, ep_num in alarm_regions:
            for r in range(1, n_rows + 1):
                fig.add_vrect(x0=alarm_start, x1=alarm_end, fillcolor='rgba(255, 0, 0, 0.12)',
                              line=dict(color='red', width=0.5, dash='dot'), layer='below', row=r, col=1)

    # ── Bottom subplot: Control actions ──
    if has_actions:
        action_sources = sorted(actions['Source'].unique())
        source_y_map = {src: i for i, src in enumerate(action_sources)}

        for _, row in actions.iterrows():
            desc = str(row['Description'])
            direction = str(row['action_direction'])

            if desc in ('SP', 'OP'):
                color = 'red' if direction == 'decrease' else ('green' if direction == 'increase' else 'grey')
            elif desc == 'MODE':
                color = 'blue'
            else:
                color = 'grey'

            symbol = 'diamond' if desc == 'SP' else ('circle' if desc == 'OP' else ('square' if desc == 'MODE' else 'x'))
            prev_val = str(row['PrevValue'])
            curr_val = str(row['Value'])
            timing = str(row['action_timing'])

            fig.add_trace(
                go.Scatter(
                    x=[row['VT_Start']], y=[source_y_map[row['Source']]], mode='markers',
                    marker=dict(color=color, size=10, symbol=symbol,
                                line=dict(width=1, color='darkgrey')),
                    showlegend=False,
                    hovertemplate=(
                        f'<b>{row["Source"]}</b> ({desc})<br>'
                        f'Time: %{{x}}<br>Direction: {direction}<br>'
                        f'{prev_val} → {curr_val}<br>'
                        f'Timing: {timing}<br><extra></extra>'
                    ),
                ), row=2, col=1)

        fig.update_yaxes(tickvals=list(source_y_map.values()), ticktext=list(source_y_map.keys()),
                         row=2, col=1, title_text='Tag', gridcolor='rgba(200,200,200,0.3)')

        # Action color legend entries
        for label, clr, sym in [('SP Increase', 'green', 'diamond'), ('SP Decrease', 'red', 'diamond'),
                                 ('OP Increase', 'green', 'circle'), ('OP Decrease', 'red', 'circle'),
                                 ('MODE Change', 'blue', 'square'), ('Other Action', 'grey', 'x')]:
            fig.add_trace(go.Scatter(x=[None], y=[None], mode='markers',
                marker=dict(color=clr, size=10, symbol=sym, line=dict(width=1, color='darkgrey')),
                name=label, legendgroup='action_legend', legendgrouptitle_text='Actions'))

    # ── Layout ──
    fig.update_layout(
        title=title, height=750 if has_actions else 550, template='plotly_white',
        hovermode='closest',
        legend=dict(groupclick='toggleitem', tracegroupgap=3, font=dict(size=10)),
        xaxis=dict(title=''))
    fig.update_yaxes(showticklabels=False, title_text='', row=1, col=1)

    return fig

In [ ]:
# Generate per-cluster plots and data exports
EPISODE_RESULTS_DIR = RESULTS_DIR / 'episode_visualizations'
EPISODE_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Only include alarm threshold line if TARGET_ALARM_LIMIT is set
target_limits = None
if TARGET_ALARM_LIMIT is not None:
    target_limits = {'lower': TARGET_PV_LOWER, 'upper': TARGET_PV_UPPER, 'alarm': TARGET_ALARM_LIMIT}
cluster_ids = sorted(cluster_info.keys())

# Build navigation dropdown HTML (shared across all plots)
def build_nav_html(all_ids, current_id, cluster_info_dict):
    """Build an HTML navigation bar with dropdown to jump between episodes."""
    options = []
    for eid in all_ids:
        cdata = cluster_info_dict[eid]
        label = (f'Episode {eid} | {cdata["total_alarms"]} alarm(s) | '
                 f'{cdata["cluster_start"].strftime("%Y-%m-%d %H:%M")}')
        selected = ' selected' if eid == current_id else ''
        options.append(f'<option value="{eid}"{selected}>{label}</option>')
    
    # Prev/Next IDs
    idx = all_ids.index(current_id)
    prev_id = all_ids[idx - 1] if idx > 0 else None
    next_id = all_ids[idx + 1] if idx < len(all_ids) - 1 else None
    
    prev_btn = (f'<button onclick="navigateTo({prev_id})" style="padding:6px 14px;cursor:pointer;'
                f'border:1px solid #ccc;border-radius:4px;background:#f8f8f8;">◀ Prev ({prev_id})</button>'
                if prev_id else '<button disabled style="padding:6px 14px;border:1px solid #eee;'
                'border-radius:4px;background:#f0f0f0;color:#aaa;">◀ Prev</button>')
    next_btn = (f'<button onclick="navigateTo({next_id})" style="padding:6px 14px;cursor:pointer;'
                f'border:1px solid #ccc;border-radius:4px;background:#f8f8f8;">Next ({next_id}) ▶</button>'
                if next_id else '<button disabled style="padding:6px 14px;border:1px solid #eee;'
                'border-radius:4px;background:#f0f0f0;color:#aaa;">Next ▶</button>')
    
    options_str = '\n'.join(options)
    nav_html = f'''
    <div style="position:sticky;top:0;z-index:9999;background:#fff;padding:10px 15px;
                border-bottom:2px solid #ddd;display:flex;align-items:center;gap:12px;
                font-family:Arial,sans-serif;font-size:14px;">
        <span style="font-weight:bold;color:#333;">Navigate:</span>
        {prev_btn}
        <select id="episode-nav" onchange="navigateTo(this.value)"
                style="padding:6px 10px;border:1px solid #ccc;border-radius:4px;
                       font-size:13px;min-width:350px;">
            {options_str}
        </select>
        {next_btn}
        <span style="color:#888;font-size:12px;margin-left:auto;">
            Episode {idx + 1} of {len(all_ids)}
        </span>
    </div>
    <script>
    function navigateTo(episodeId) {{
        var padded = String(episodeId).padStart(4, '0');
        var url = '../episode_' + padded + '/episode_' + padded + '_plot.html';
        window.location.href = url;
    }}
    </script>
    '''
    return nav_html

print(f'Generating plots and data for {len(cluster_ids)} alarm clusters...')
print(f'Target tag: {TARGET_TAG} | Window buffer: -{BUFFER_BEFORE_MINUTES} min / +{BUFFER_AFTER_MINUTES} min')
print(f'Output directory: {EPISODE_RESULTS_DIR}/')

for i, cid in enumerate(cluster_ids):
    cdata = cluster_info[cid]
    cactions = cluster_actions.get(cid, None)

    c_start = cdata['cluster_start']
    c_end = cdata['cluster_end']
    window_start = c_start - pd.Timedelta(minutes=BUFFER_BEFORE_MINUTES)
    window_end = c_end + pd.Timedelta(minutes=BUFFER_AFTER_MINUTES)
    n_alarms = cdata['total_alarms']
    c_type = cdata['cluster_type']

    # Create episode folder
    ep_folder = EPISODE_RESULTS_DIR / f'episode_{cid:04d}'
    ep_folder.mkdir(parents=True, exist_ok=True)

    title = (f'{TARGET_TAG} | Episode {cid} | {n_alarms} alarm(s) | {c_type} | '
             f'{c_start.strftime("%Y-%m-%d %H:%M")} to {c_end.strftime("%Y-%m-%d %H:%M")}')

    # 1. Generate HTML plot with navigation
    fig = create_window_plot(
        window_start, window_end, op_pv_data_df, ordered_tags, base_colors,
        title=title, actions=cactions,
        cluster_regions=[(cid, c_start, c_end)],
        alarm_regions=cdata['alarms'],
        target_limits=target_limits,
        operating_limits=tag_operating_limits)

    if fig is not None:
        # Generate plotly HTML and inject navigation bar
        plot_html = fig.to_html(include_plotlyjs='cdn', full_html=True)
        nav_html = build_nav_html(cluster_ids, cid, cluster_info)
        # Insert nav bar right after <body> tag
        plot_html = plot_html.replace('<body>', f'<body>\n{nav_html}', 1)
        
        out_path = ep_folder / f'episode_{cid:04d}_plot.html'
        with open(out_path, 'w') as f:
            f.write(plot_html)

    # 2. Export minutewise PV data for all tags in the window
    pv_mask = (op_pv_data_df.index >= window_start) & (op_pv_data_df.index <= window_end)
    pv_window = op_pv_data_df.loc[pv_mask].copy()
    if not pv_window.empty:
        pv_window.to_csv(ep_folder / f'episode_{cid:04d}_pv_data.csv')

    # 3. Export events corresponding to this episode window
    events_mask = (
        (raw_change_events['VT_Start'] >= window_start) &
        (raw_change_events['VT_Start'] <= window_end)
    )
    ep_events = raw_change_events.loc[events_mask].copy()
    if not ep_events.empty:
        ep_events.to_csv(ep_folder / f'episode_{cid:04d}_events.csv', index=False)

    if (i + 1) % 50 == 0 or (i + 1) == len(cluster_ids):
        print(f'  {i + 1}/{len(cluster_ids)} episodes done')

print(f'\nAll episode data saved to {EPISODE_RESULTS_DIR}/')

Generating plots and data for 594 alarm clusters...
Target tag: 03LIC_1016 | Window buffer: -240 min / +60 min
Output directory: ../RESULTS/03LIC_1016_episodes/episode_visualizations/
  50/594 episodes done
  100/594 episodes done
  150/594 episodes done
  200/594 episodes done
  250/594 episodes done
  No PV/OP data in window 2022-12-19 19:51:03.602000 to 2022-12-20 01:50:16.403000
  No PV/OP data in window 2022-12-20 00:24:42.353000 to 2022-12-20 06:03:01.153000
  No PV/OP data in window 2022-12-20 01:43:28.154000 to 2022-12-20 07:11:15.203000
  300/594 episodes done
  350/594 episodes done
  400/594 episodes done
  450/594 episodes done
  500/594 episodes done
  550/594 episodes done
  No PV/OP data in window 2025-06-24 06:37:47.411000 to 2025-06-24 11:38:52.094000
  No PV/OP data in window 2025-06-24 11:16:24.031000 to 2025-06-24 16:17:04.099000
  No PV/OP data in window 2025-06-24 14:38:01.428000 to 2025-06-24 19:38:39.215000
  No PV/OP data in window 2025-06-25 09:44:02.134000 to

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SELF-CONTAINED CUSTOM TIME WINDOW PLOT
# This cell runs standalone — no need to execute any prior cells first.
# Heavy data loading is cached in the kernel, so re-running with a new
# START_TIME / END_TIME (without changing the config below) is fast.
# ═══════════════════════════════════════════════════════════════════════════════

# ── Time window to plot (edit these) ──
START_TIME = '2024-06-29 08:00'
END_TIME   = '2024-06-29 23:59'

# ── Configuration (must match the tag/files you want to visualize) ──
TARGET_TAG = '03LIC_1071'           # Base tag name (e.g., '03LIC_1071', '03LIC_1016')
TARGET_PV_COL = f'{TARGET_TAG}.PV'  # PV column name in time series data
TARGET_ALARM_LIMIT = None           # None = auto-detect from events; set a number to override
ALARM_CONDITION = 'PVLO'            # Alarm condition used for alarm-limit auto-detection

PV_OP_FILE = '/home/h604827/ControlActions/DATA/PV-OP_data/03LIC_1071_JAN_2026.parquet'   # PV/OP time series parquet
EVENTS_FILE = '/home/h604827/ControlActions/DATA/combined_events/03LIC_1071_PVLO_PVHI_combined_events.parquet'  # Events data
RESULTS_DIR_NAME = f'{TARGET_TAG}_episodes'
CLUSTERS_FILE = f'/home/h604827/ControlActions/RESULTS/03LIC_1071_PVLO_episodes_12JUN2026_1219/03LIC_1071_pvlo_alarms_clustered_with_control_actions.xlsx'
TRIP_FILE = '../DATA/Final_List_Trip_Duration.csv'
FILTER_TRIPS = True
OPERATING_LIMITS_FILE = '../DATA/operating_limits.xlsx'

# ── Imports ──
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots


# ═══════════════════════════════════════════════════════════════════════════════
# Plot helper (always defined so it is available standalone)
# ═══════════════════════════════════════════════════════════════════════════════
def create_window_plot(window_start, window_end, op_pv_df, ordered_tags, base_colors,
                       title='', actions=None, cluster_regions=None, alarm_regions=None,
                       target_limits=None, operating_limits=None,
                       target_tag=None, target_pv_col=None):
    """
    General-purpose plot for any time window.

    Parameters:
        window_start, window_end: pd.Timestamp - time window boundaries
        op_pv_df: DataFrame with TimeStamp index and PV/OP columns
        ordered_tags: list of column names to plot
        base_colors: dict mapping base tag name to color
        title: str - plot title
        actions: DataFrame of control actions (needs Source, Description, action_direction,
                 VT_Start, PrevValue, Value, action_timing columns). None if no actions.
        cluster_regions: list of (cluster_id, start, end) tuples for orange shading
        alarm_regions: list of (alarm_start, alarm_end, episode_num) tuples for red shading
        target_limits: dict with 'lower', 'upper', 'alarm' for target PV horizontal lines
        operating_limits: dict {tag_name: {'upper': val, 'lower': val}} for per-tag limit lines
        target_tag: str - base tag name to highlight (e.g., '03LIC_1016')
        target_pv_col: str - full PV column name (e.g., '03LIC_1016.PV')
    """
    # Use module-level defaults if not provided
    if target_tag is None:
        target_tag = TARGET_TAG
    if target_pv_col is None:
        target_pv_col = TARGET_PV_COL

    # Extract PV/OP data for window
    mask = (op_pv_df.index >= window_start) & (op_pv_df.index <= window_end)
    window_df = op_pv_df.loc[mask, [c for c in ordered_tags if c in op_pv_df.columns]].copy()

    if window_df.empty:
        print(f'  No PV/OP data in window {window_start} to {window_end}')
        return None

    has_actions = actions is not None and len(actions) > 0

    # Determine subplot heights
    if has_actions:
        n_action_tags = actions['Source'].nunique()
        action_height = max(0.15, min(0.35, n_action_tags * 0.04))
        row_heights = [1 - action_height, action_height]
        fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.05,
                            row_heights=row_heights,
                            subplot_titles=('PV / OP Trends (normalized)', 'Control Actions'))
    else:
        fig = make_subplots(rows=1, cols=1,
                            subplot_titles=('PV / OP Trends (normalized)',))

    # ── Top subplot: PV/OP traces (min-max normalized) ──
    x_range = [window_df.index[0], window_df.index[-1]]

    for col in ordered_tags:
        if col not in window_df.columns:
            continue
        series = window_df[col]
        if series.isna().all():
            continue

        col_min, col_max = series.min(), series.max()
        if col_max == col_min:
            normalized = pd.Series(0.5, index=series.index)
        else:
            normalized = (series - col_min) / (col_max - col_min)

        base_tag = col.rsplit('.', 1)[0]
        suffix = col.rsplit('.', 1)[1]
        color = base_colors.get(base_tag, '#888888')
        is_op = suffix == 'OP'
        is_target = base_tag == target_tag

        fig.add_trace(
            go.Scatter(
                x=series.index, y=normalized, mode='lines', name=col,
                legendgroup=base_tag,
                legendgrouptitle_text=base_tag if suffix == 'PV' else None,
                line=dict(color=color, width=2.5 if is_target else 1.5,
                          dash='dot' if is_op else 'solid'),
                visible=True if is_target else 'legendonly',
                customdata=np.column_stack([series.values]),
                hovertemplate=(f'<b>{col}</b><br>Time: %{{x}}<br>'
                               'Value: %{customdata[0]:.4f}<br><extra></extra>'),
            ),
            row=1, col=1
        )

        # ── Operating limit lines for this PV tag (upper + lower as single legend item) ──
        if operating_limits and suffix == 'PV' and col in operating_limits and col_max != col_min:
            limits = operating_limits[col]
            upper_val = limits['upper']
            lower_val = limits['lower']
            norm_upper = (upper_val - col_min) / (col_max - col_min)
            norm_lower = (lower_val - col_min) / (col_max - col_min)
            # Upper limit line (shows legend entry for both)
            fig.add_trace(
                go.Scatter(
                    x=x_range, y=[norm_upper, norm_upper], mode='lines',
                    name=f'{col} limits ({lower_val:.2f}–{upper_val:.2f})',
                    legendgroup=base_tag,
                    line=dict(color=color, width=1.2, dash='dash'),
                    visible=True if is_target else 'legendonly',
                    hovertemplate=f'<b>{col} upper limit</b><br>Value: {upper_val:.4f}<extra></extra>',
                ),
                row=1, col=1
            )
            # Lower limit line (no legend entry, tied to same group)
            fig.add_trace(
                go.Scatter(
                    x=x_range, y=[norm_lower, norm_lower], mode='lines',
                    name=f'{col} limits',
                    legendgroup=base_tag,
                    line=dict(color=color, width=1.2, dash='dash'),
                    visible=True if is_target else 'legendonly',
                    showlegend=False,
                    hovertemplate=f'<b>{col} lower limit</b><br>Value: {lower_val:.4f}<extra></extra>',
                ),
                row=1, col=1
            )

    # ── Horizontal limit lines for target PV (alarm threshold) ──
    if target_limits and target_pv_col in window_df.columns:
        pv_series = window_df[target_pv_col]
        pv_min, pv_max = pv_series.min(), pv_series.max()
        if pv_max != pv_min:
            # Only add alarm threshold line (operating limits already handled above)
            alarm_val = target_limits['alarm']
            norm_val = (alarm_val - pv_min) / (pv_max - pv_min)
            fig.add_trace(
                go.Scatter(
                    x=x_range, y=[norm_val, norm_val], mode='lines',
                    name=f'Alarm ({alarm_val})',
                    legendgroup=target_tag,
                    line=dict(color='red', width=1.5, dash='dash'),
                    visible=True,
                    hovertemplate=f'<b>{target_pv_col} alarm threshold</b><br>Value: {alarm_val}<extra></extra>',
                ),
                row=1, col=1
            )

    # ── Shading ──
    n_rows = 2 if has_actions else 1

    # Cluster-level shading (light orange)
    if cluster_regions:
        for cid_r, c_start_r, c_end_r in cluster_regions:
            for r in range(1, n_rows + 1):
                fig.add_vrect(x0=c_start_r, x1=c_end_r, fillcolor='rgba(255, 165, 0, 0.10)',
                              line=dict(width=0), layer='below', row=r, col=1)
                fig.add_vline(x=c_start_r, line=dict(color='orange', width=1.5, dash='dash'), row=r, col=1)
                fig.add_vline(x=c_end_r, line=dict(color='orange', width=1.5, dash='dash'), row=r, col=1)

    # Individual alarm shading (light red)
    if alarm_regions:
        for alarm_start, alarm_end, ep_num in alarm_regions:
            for r in range(1, n_rows + 1):
                fig.add_vrect(x0=alarm_start, x1=alarm_end, fillcolor='rgba(255, 0, 0, 0.12)',
                              line=dict(color='red', width=0.5, dash='dot'), layer='below', row=r, col=1)

    # ── Bottom subplot: Control actions ──
    if has_actions:
        action_sources = sorted(actions['Source'].unique())
        source_y_map = {src: i for i, src in enumerate(action_sources)}

        for _, row in actions.iterrows():
            desc = str(row['Description'])
            direction = str(row['action_direction'])

            if desc in ('SP', 'OP'):
                color = 'red' if direction == 'decrease' else ('green' if direction == 'increase' else 'grey')
            elif desc == 'MODE':
                color = 'blue'
            else:
                color = 'grey'

            symbol = 'diamond' if desc == 'SP' else ('circle' if desc == 'OP' else ('square' if desc == 'MODE' else 'x'))
            prev_val = str(row['PrevValue'])
            curr_val = str(row['Value'])
            timing = str(row['action_timing'])

            fig.add_trace(
                go.Scatter(
                    x=[row['VT_Start']], y=[source_y_map[row['Source']]], mode='markers',
                    marker=dict(color=color, size=10, symbol=symbol,
                                line=dict(width=1, color='darkgrey')),
                    showlegend=False,
                    hovertemplate=(
                        f'<b>{row["Source"]}</b> ({desc})<br>'
                        f'Time: %{{x}}<br>Direction: {direction}<br>'
                        f'{prev_val} → {curr_val}<br>'
                        f'Timing: {timing}<br><extra></extra>'
                    ),
                ), row=2, col=1)

        fig.update_yaxes(tickvals=list(source_y_map.values()), ticktext=list(source_y_map.keys()),
                         row=2, col=1, title_text='Tag', gridcolor='rgba(200,200,200,0.3)')

        # Action color legend entries
        for label, clr, sym in [('SP Increase', 'green', 'diamond'), ('SP Decrease', 'red', 'diamond'),
                                 ('OP Increase', 'green', 'circle'), ('OP Decrease', 'red', 'circle'),
                                 ('MODE Change', 'blue', 'square'), ('Other Action', 'grey', 'x')]:
            fig.add_trace(go.Scatter(x=[None], y=[None], mode='markers',
                marker=dict(color=clr, size=10, symbol=sym, line=dict(width=1, color='darkgrey')),
                name=label, legendgroup='action_legend', legendgrouptitle_text='Actions'))

    # ── Layout ──
    fig.update_layout(
        title=title, height=750 if has_actions else 550, template='plotly_white',
        hovermode='closest',
        legend=dict(groupclick='toggleitem', tracegroupgap=3, font=dict(size=10)),
        xaxis=dict(title=''))
    fig.update_yaxes(showticklabels=False, title_text='', row=1, col=1)

    return fig


# ═══════════════════════════════════════════════════════════════════════════════
# Load data once (cached by config signature; re-runs reuse loaded data)
# ═══════════════════════════════════════════════════════════════════════════════
_setup_sig = (TARGET_TAG, ALARM_CONDITION, PV_OP_FILE, EVENTS_FILE, CLUSTERS_FILE, FILTER_TRIPS)
if globals().get('_custom_plot_sig') != _setup_sig:
    print('Loading data for custom window plot...')

    # ── Load PV/OP time series ──
    op_pv_data_df = pd.read_parquet(PV_OP_FILE)
    if 'TimeStamp' in op_pv_data_df.columns:
        op_pv_data_df['TimeStamp'] = pd.to_datetime(op_pv_data_df['TimeStamp'])
        op_pv_data_df.set_index('TimeStamp', inplace=True)
    op_pv_data_df.sort_index(inplace=True)

    # ── Trip period filtering on PV/OP data ──
    trips_df = None
    if FILTER_TRIPS:
        trips_df = pd.read_csv(TRIP_FILE)
        trips_df['Stop Date'] = pd.to_datetime(trips_df['Stop Date'])
        trips_df['Start Date'] = pd.to_datetime(trips_df['Start Date'])
        pre_trip = len(op_pv_data_df)
        trip_mask = pd.Series(False, index=op_pv_data_df.index)
        for _, trip in trips_df.iterrows():
            trip_mask |= (op_pv_data_df.index >= trip['Stop Date']) & (op_pv_data_df.index <= trip['Start Date'])
        op_pv_data_df = op_pv_data_df[~trip_mask]
        print(f'Trip filtering (PV/OP): {pre_trip:,} → {len(op_pv_data_df):,} rows')

    # ── Load alarm clusters ──
    alarms_df = pd.read_excel(CLUSTERS_FILE, sheet_name=0)

    # ── Load raw events ──
    if EVENTS_FILE.endswith('.parquet'):
        raw_events_df = pd.read_parquet(EVENTS_FILE)
    else:
        raw_events_df = pd.read_csv(EVENTS_FILE, low_memory=False)
    raw_events_df['VT_Start'] = pd.to_datetime(raw_events_df['VT_Start'])
    raw_events_df = raw_events_df.sort_values('VT_Start').reset_index(drop=True)

    # ── Auto-detect alarm limit from events data ──
    _DETECTED_ALARM_LIMIT = None
    if 'AlarmLimit' in raw_events_df.columns:
        _alarm_start_rows = raw_events_df[
            (raw_events_df['Source'] == TARGET_TAG) &
            (raw_events_df['ConditionName'] == ALARM_CONDITION) &
            (raw_events_df['Category'] == 1) &
            (raw_events_df['Action'].isna() | (raw_events_df['Action'] == ''))
        ]
        _alarm_limits = _alarm_start_rows['AlarmLimit'].dropna().unique()
        if len(_alarm_limits) >= 1:
            _DETECTED_ALARM_LIMIT = float(_alarm_limits[0])
            print(f'Auto-detected alarm limit: {_DETECTED_ALARM_LIMIT}')

    # ── Trip filter events ──
    if FILTER_TRIPS and trips_df is not None:
        ev_trip_mask = pd.Series(False, index=raw_events_df.index)
        for _, trip in trips_df.iterrows():
            ev_trip_mask |= (raw_events_df['VT_Start'] >= trip['Stop Date']) & (raw_events_df['VT_Start'] <= trip['Start Date'])
        raw_events_df = raw_events_df[~ev_trip_mask].reset_index(drop=True)

    # Pre-filter to CHANGE events only
    raw_change_events = raw_events_df[raw_events_df['ConditionName'] == 'CHANGE'].copy()

    # ── Load operating limits ──
    op_limits_raw = pd.read_excel(OPERATING_LIMITS_FILE)
    tag_operating_limits = {}  # {tag_name: {'upper': val, 'lower': val}}
    for _, row in op_limits_raw.iterrows():
        tag = row['TagName']
        upper = row['NEW_UPPER_LIMIT']
        lower = row['NEW_LOWER_LIMIT']
        if pd.isna(upper) or (isinstance(upper, str) and 'NOT' in upper.upper()):
            upper = row['OLD_UPPER_LIMIT']
        if pd.isna(lower) or (isinstance(lower, str) and 'NOT' in lower.upper()):
            lower = row['OLD_LOWER_LIMIT']
        try:
            tag_operating_limits[tag] = {'upper': float(upper), 'lower': float(lower)}
        except (ValueError, TypeError):
            pass

    TARGET_PV_LOWER = tag_operating_limits.get(TARGET_PV_COL, {}).get('lower', None)
    TARGET_PV_UPPER = tag_operating_limits.get(TARGET_PV_COL, {}).get('upper', None)

    # ── Identify tag columns and build ordered tag list (PV then OP per base) ──
    tag_cols = [c for c in op_pv_data_df.columns if c.endswith('.PV') or c.endswith('.OP')]
    pv_cols = sorted([c for c in tag_cols if c.endswith('.PV')])
    op_cols = sorted([c for c in tag_cols if c.endswith('.OP')])
    pv_bases = {c.replace('.PV', ''): c for c in pv_cols}
    op_bases = {c.replace('.OP', ''): c for c in op_cols}
    all_bases = sorted(set(list(pv_bases.keys()) + list(op_bases.keys())))
    ordered_tags = []
    for base in all_bases:
        if base in pv_bases:
            ordered_tags.append(pv_bases[base])
        if base in op_bases:
            ordered_tags.append(op_bases[base])

    # ── Color palette per base tag (PV and OP of same base share a color) ──
    palette = (px.colors.qualitative.Dark24 + px.colors.qualitative.Light24 +
               px.colors.qualitative.Alphabet)
    base_colors = {base: palette[i % len(palette)] for i, base in enumerate(all_bases)}

    # ── Build cluster info: one entry per cluster with its individual alarms ──
    cluster_info = {}
    for cid, grp in alarms_df.groupby('cluster_id'):
        cluster_info[cid] = {
            'cluster_start': grp['cluster_start_time'].iloc[0],
            'cluster_end': grp['cluster_end_time'].iloc[0],
            'cluster_type': grp['cluster_type'].iloc[0],
            'total_alarms': grp['cluster_total_alarms'].iloc[0],
            'alarms': list(grp[['alarm_start', 'alarm_end', 'episode_num']].itertuples(index=False, name=None)),
        }

    _custom_plot_sig = _setup_sig
    print(f'Ready: {len(op_pv_data_df):,} PV/OP rows, {len(raw_change_events):,} CHANGE events, '
          f'{len(cluster_info)} clusters, {len(ordered_tags)} tags')

# Effective alarm limit: user override takes precedence over auto-detected value
effective_alarm_limit = TARGET_ALARM_LIMIT if TARGET_ALARM_LIMIT is not None else _DETECTED_ALARM_LIMIT


# ═══════════════════════════════════════════════════════════════════════════════
# Custom time window plot
# ═══════════════════════════════════════════════════════════════════════════════
start_ts = pd.Timestamp(START_TIME)
end_ts = pd.Timestamp(END_TIME)

# Find alarm clusters that overlap with this window
overlapping_alarms = []
overlapping_clusters = []
for cid, cdata in cluster_info.items():
    c_start = cdata['cluster_start']
    c_end = cdata['cluster_end']
    if c_start <= end_ts and c_end >= start_ts:
        overlapping_clusters.append((cid, c_start, c_end))
        for alarm_start, alarm_end, ep_num in cdata['alarms']:
            if alarm_start <= end_ts and alarm_end >= start_ts:
                overlapping_alarms.append((alarm_start, alarm_end, ep_num))

# Get control actions from raw events (not the clustered Excel)
window_actions = raw_change_events[
    (raw_change_events['VT_Start'] >= start_ts) & (raw_change_events['VT_Start'] <= end_ts)
].copy()

if len(window_actions) > 0:
    # Compute action_direction from Value vs PrevValue
    _val_num = pd.to_numeric(window_actions['Value'], errors='coerce')
    _prev_num = pd.to_numeric(window_actions['PrevValue'], errors='coerce')
    window_actions['action_direction'] = np.where(
        _val_num > _prev_num, 'increase',
        np.where(_val_num < _prev_num, 'decrease', 'no_change'))

    # Compute action_timing relative to overlapping clusters (if any)
    def get_timing(ts):
        for _, c_start, c_end in overlapping_clusters:
            if ts < c_start:
                return 'before'
            elif ts > c_end:
                return 'after'
            else:
                return 'during'
        return 'no_cluster'
    window_actions['action_timing'] = window_actions['VT_Start'].apply(get_timing)
else:
    window_actions = None

print(f'Window: {start_ts} to {end_ts}')
print(f'Overlapping clusters: {len(overlapping_clusters)}, Individual alarms: {len(overlapping_alarms)}')
print(f'Control actions in window (from raw events): {len(window_actions) if window_actions is not None else 0}')

target_limits = None
if effective_alarm_limit is not None:
    target_limits = {'lower': TARGET_PV_LOWER, 'upper': TARGET_PV_UPPER, 'alarm': effective_alarm_limit}
fig = create_window_plot(
    start_ts, end_ts, op_pv_data_df, ordered_tags, base_colors,
    title=f'Custom Window: {START_TIME} to {END_TIME}',
    actions=window_actions,
    cluster_regions=overlapping_clusters if overlapping_clusters else None,
    alarm_regions=overlapping_alarms if overlapping_alarms else None,
    target_limits=target_limits,
    operating_limits=tag_operating_limits)

if fig is not None:
    fig.show()


Loading data for custom window plot...
Trip filtering (PV/OP): 1,737,586 → 1,718,039 rows
Auto-detected alarm limit: 28.75
Ready: 1,718,039 PV/OP rows, 314,968 CHANGE events, 539 clusters, 43 tags
Window: 2024-06-29 08:00:00 to 2024-06-29 23:59:00
Overlapping clusters: 2, Individual alarms: 2
Control actions in window (from raw events): 144


In [ ]:
# Check raw events for 03PIC_1013 control actions between May 5, 2025 06:00 - 12:00
raw_events = pd.read_csv(DATA_DIR / 'df_df_events_1071_export.csv', low_memory=False)
raw_events['VT_Start'] = pd.to_datetime(raw_events['VT_Start'])

mask = (
    raw_events['Source'].str.contains('1071', na=False) &
    (raw_events['VT_Start'] >= '2025-01-08 02:00') &
    (raw_events['VT_Start'] <= '2025-01-08 04:00')
)
result = raw_events.loc[mask, ['Source', 'ConditionName', 'Description', 'PrevValue', 'Value', 'VT_Start', 'Category']].copy()
result = result.sort_values('VT_Start')
print(f'Found {len(result)} events for *1013* between 2025-05-05 06:00 and 12:00:\n')
result

Found 31 events for *1013* between 2025-05-05 06:00 and 12:00:



,Source,ConditionName,Description,PrevValue,Value,VT_Start,Category
54205,03LIC_1071,PVLO,3E107 LEVEL,NaN,28.647,2025-01-08 02:51:46.114100,1
54214,03LIC_1071,PVLO,PVLO LOW 3E107 LEVEL ...,NaN,NaN,2025-01-08 02:52:19.859900,7
54218,03LIC_1071,PVLO,3E107 LEVEL,NaN,31.769,2025-01-08 02:52:20.031200,1
54226,03LIC_1071,NaN,3E107 LEVEL MODE AUTO ...,NaN,NaN,2025-01-08 03:00:40.813000,7
632273,03LIC_1071,CHANGE,MODE,AUTO,MAN,2025-01-08 03:00:40.813000,7
54223,03LIC_1071,CHANGE,MODE,NaN,MAN,2025-01-08 03:00:40.813000,7
54232,03LIC_1071,CHANGE,OP,NaN,58.0000,2025-01-08 03:00:44.933000,7
54235,03LIC_1071,NaN,3E107 LEVEL OP 52.280...,NaN,NaN,2025-01-08 03:00:44.933000,7
632277,03LIC_1071,CHANGE,OP,52.2807,58.0000,2025-01-08 03:00:44.933000,7
54244,03LIC_1071,CHANGE,OP,NaN,52.0000,2025-01-08 03:02:43.849900,7


In [ ]:
# Find clusters with continuous stretches of target PV <= 0
ZERO_THRESHOLD = 0
LONG_STRETCH_MIN = 30  # minutes

clusters_agg = alarms_df.groupby('cluster_id').agg(
    cluster_start=('cluster_start_time', 'first'),
    cluster_end=('cluster_end_time', 'first')
)

long_stretches = []   # >= 30 min
short_stretches = []  # < 30 min but > 0

for cid, row in clusters_agg.iterrows():
    ws = row['cluster_start'] - pd.Timedelta(minutes=30)
    we = row['cluster_end'] + pd.Timedelta(minutes=30)
    
    if TARGET_PV_COL not in op_pv_data_df.columns:
        print(f'WARNING: {TARGET_PV_COL} not in PV/OP data columns')
        break
    
    window = op_pv_data_df.loc[ws:we, TARGET_PV_COL]
    if window.empty:
        continue
    
    below_zero = (window <= ZERO_THRESHOLD).astype(int)
    if below_zero.sum() == 0:
        continue
    
    # Group consecutive runs
    groups = (below_zero != below_zero.shift()).cumsum()
    
    for _, grp in window.groupby(groups):
        if (grp <= ZERO_THRESHOLD).all() and len(grp) > 1:
            dur_min = (grp.index[-1] - grp.index[0]).total_seconds() / 60
            info = (cid, dur_min, grp.min(), grp.index[0], grp.index[-1])
            if dur_min >= LONG_STRETCH_MIN:
                long_stretches.append(info)
            elif dur_min > 0:
                short_stretches.append(info)

print(f'Target: {TARGET_PV_COL} | Total clusters checked: {len(clusters_agg)}')
print(f'\n{"="*70}')
print(f'Clusters with >= {LONG_STRETCH_MIN} min continuous PV <= {ZERO_THRESHOLD}:  '
      f'{len(set(c[0] for c in long_stretches))} clusters, {len(long_stretches)} stretches')
print(f'{"="*70}')
for cid, dur, minv, s, e in sorted(long_stretches):
    print(f'  Cluster {cid:>4d}: {dur:>6.0f} min | min PV = {minv:.2f} | {s} → {e}')

print(f'\n{"="*70}')
print(f'Clusters with < {LONG_STRETCH_MIN} min continuous PV <= {ZERO_THRESHOLD}:  '
      f'{len(set(c[0] for c in short_stretches))} clusters, {len(short_stretches)} stretches')
print(f'{"="*70}')
for cid, dur, minv, s, e in sorted(short_stretches):
    print(f'  Cluster {cid:>4d}: {dur:>6.0f} min | min PV = {minv:.2f} | {s} → {e}')

# Summary
all_affected = set(c[0] for c in long_stretches + short_stretches)
print(f'\n{"="*70}')
print(f'SUMMARY: {len(all_affected)} clusters affected out of {len(clusters_agg)} total')
print(f'  Long (>= {LONG_STRETCH_MIN} min): cluster IDs = {sorted(set(c[0] for c in long_stretches))}')
print(f'  Short (< {LONG_STRETCH_MIN} min): cluster IDs = {sorted(set(c[0] for c in short_stretches))}')

Total clusters checked: 539

Clusters with >= 30 min continuous PV <= 0:  9 clusters, 9 stretches
  Cluster   80:    100 min | min PV = -0.51 | 2022-06-21 14:29:00 → 2022-06-21 16:09:00
  Cluster  214:     45 min | min PV = -0.54 | 2023-12-26 18:09:00 → 2023-12-26 18:54:00
  Cluster  216:    152 min | min PV = -0.54 | 2023-12-27 01:52:00 → 2023-12-27 04:24:00
  Cluster  432:     34 min | min PV = -0.53 | 2024-09-24 06:28:00 → 2024-09-24 07:02:00
  Cluster  434:     34 min | min PV = -1.31 | 2024-09-26 08:27:00 → 2024-09-26 09:01:00
  Cluster  475:    160 min | min PV = -1.30 | 2025-01-08 09:19:00 → 2025-01-08 11:59:00
  Cluster  476:    257 min | min PV = -1.31 | 2025-01-08 14:50:00 → 2025-01-08 19:07:00
  Cluster  481:    101 min | min PV = -1.30 | 2025-01-12 09:51:00 → 2025-01-12 11:32:00
  Cluster  534:     72 min | min PV = -1.33 | 2025-06-21 18:29:00 → 2025-06-21 19:41:00

Clusters with < 30 min continuous PV <= 0:  13 clusters, 23 stretches
  Cluster   80:     11 min | min PV = -